# PR. 2 – Derivable Judgement
## Part B – Data Analysis & Testing Tasks

**Mathematics & Advanced Statistics**

This notebook implements the practical tasks specified in the assignment:
1. Generate a health-record dataset.
2. Formulate at least two hypotheses.
3. Calculate a confidence interval.
4. Find critical values and p-values.
5. Perform a t-test.
6. Perform a chi-square test.
7. Perform ANOVA.
8. Calculate covariance and correlation.
9. State the decision and interpretation for each test.


## 1. Importing Libraries

I used NumPy, Pandas, SciPy and Matplotlib for the practical work below.


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

np.random.seed(42)
alpha = 0.05


## 2. Creating the Dataset

The assignment asks us to generate a health-record dataset. I created 1,000 sample records using the fields given in the question.


In [ ]:
n = 1000

ages = np.random.randint(18, 76, n)
age_group = pd.cut(
    ages,
    bins=[17, 25, 35, 45, 60, 100],
    labels=["18-25", "26-35", "36-45", "46-60", "60+"]
)

gender = np.random.choice(
    ["Male", "Female", "Other"], n, p=[0.49, 0.49, 0.02]
)

region = np.random.choice(["North", "South", "East", "West"], n)

smoking = np.random.choice(
    ["Smoker", "Non-Smoker", "Former Smoker"],
    n,
    p=[0.22, 0.62, 0.16]
)

exercise = np.random.choice(
    ["Daily", "Weekly", "Rarely", "Never"],
    n,
    p=[0.20, 0.40, 0.30, 0.10]
)

bmi = np.clip(
    np.random.normal(
        24.5 + 0.035 * (ages - 40) + (smoking == "Smoker") * 1.2,
        3.5,
        n
    ),
    16,
    40
)

weight = np.clip(
    bmi * (np.random.normal(1.72, 0.09, n) ** 2),
    45,
    130
)

blood_pressure = np.clip(
    np.random.normal(
        120 + 0.45 * (ages - 40) + (smoking == "Smoker") * 4,
        15,
        n
    ),
    90,
    190
)

cholesterol = np.clip(
    np.random.normal(
        185 + 0.7 * (ages - 40) + (smoking == "Smoker") * 8,
        25,
        n
    ),
    120,
    320
)

glucose = np.clip(
    np.random.normal(
        92 + 0.55 * (ages - 40) + (bmi - 24) * 2,
        15,
        n
    ),
    65,
    220
)

# Diabetes probability
logit_diabetes = (
    -4.0
    + 0.055 * (ages - 40)
    + 0.13 * (bmi - 24)
    + 0.45 * (smoking == "Smoker")
    - 0.25 * (exercise == "Daily")
    - 0.12 * (exercise == "Weekly")
)

p_diabetes = 1 / (1 + np.exp(-logit_diabetes))
diabetes = np.random.rand(n) < p_diabetes

# Hypertension probability
logit_hypertension = (
    -2.0
    + 0.065 * (ages - 40)
    + 0.09 * (bmi - 24)
    + 0.40 * (smoking == "Smoker")
)

p_hypertension = 1 / (1 + np.exp(-logit_hypertension))
hypertension = np.random.rand(n) < p_hypertension

visit_dates = pd.to_datetime("2024-01-01") + pd.to_timedelta(
    np.random.randint(0, 730, n), unit="D"
)

df = pd.DataFrame({
    "record_id": [f"HR-{i:04d}" for i in range(1, n + 1)],
    "age_group": age_group.astype(str),
    "age": ages,
    "weight": weight.round(1),
    "gender": gender,
    "region": region,
    "smoking_status": smoking,
    "exercise_frequency": exercise,
    "bmi": bmi.round(2),
    "blood_pressure": blood_pressure.round(1),
    "diabetes": diabetes,
    "hypertension": hypertension,
    "cholesterol_level": cholesterol.round(1),
    "glucose_level": glucose.round(1),
    "visit_date": visit_dates.date
})

df.head()


## 3. Checking the Dataset

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nBasic statistics:")
display(df.describe(include="all").T)


## 4. Saving the Dataset

I am saving the generated data as a CSV file so it can also be checked separately.


In [ ]:
df.to_csv("health_records_dataset.csv", index=False)
print("Saved as health_records_dataset.csv")


## 5. Hypotheses

### Hypothesis 1 – Smoking and Diabetes
- **H₀:** Smoking status has no significant association with diabetes prevalence.
- **H₁:** Smoking status has a significant association with diabetes prevalence.
- **Test:** Chi-square test of independence.

### Hypothesis 2 – Age Groups and Diabetes
- **H₀:** There is no significant difference in diabetes rate among the age groups.
- **H₁:** At least one age group has a different diabetes rate.
- **Test:** One-way ANOVA.

### Additional Hypothesis – BMI
- **H₀:** Mean BMI is equal for smokers and non-smokers.
- **H₁:** Mean BMI is different for smokers and non-smokers.
- **Test:** Independent two-sample Welch t-test.

**Significance level:** α = 0.05


## 6. Confidence Interval for Mean Age

In [ ]:
mean_age = df["age"].mean()
sd_age = df["age"].std(ddof=1)
n_age = df["age"].count()
se_age = sd_age / np.sqrt(n_age)

ci_low, ci_high = stats.t.interval(
    confidence=0.95,
    df=n_age - 1,
    loc=mean_age,
    scale=se_age
)

print(f"Sample size: {n_age}")
print(f"Mean age: {mean_age:.3f}")
print(f"Sample standard deviation: {sd_age:.3f}")
print(f"95% Confidence Interval: ({ci_low:.3f}, {ci_high:.3f})")


### My interpretation
The confidence interval gives the range in which the population mean age is estimated to fall, based on this sample.


## 7. Critical Value and p-value

In [ ]:
print("Significance level (alpha):", alpha)
print("Decision rule:")
print("If p-value < 0.05 -> Reject H0")
print("If p-value >= 0.05 -> Fail to reject H0")


## 8. t-test – BMI of Smokers and Non-Smokers

In [ ]:
smokers = df.loc[
    df["smoking_status"] == "Smoker", "bmi"
]

non_smokers = df.loc[
    df["smoking_status"] == "Non-Smoker", "bmi"
]

t_result = stats.ttest_ind(
    smokers,
    non_smokers,
    equal_var=False
)

t_stat = t_result.statistic
t_p = t_result.pvalue
t_df = t_result.df

t_critical = stats.t.ppf(
    1 - alpha / 2,
    t_df
)

print(f"Smokers: n={len(smokers)}, mean BMI={smokers.mean():.3f}")
print(f"Non-smokers: n={len(non_smokers)}, mean BMI={non_smokers.mean():.3f}")
print(f"t-statistic = {t_stat:.4f}")
print(f"Welch degrees of freedom = {t_df:.2f}")
print(f"Critical t-value = ±{t_critical:.4f}")
print(f"p-value = {t_p:.8g}")

if t_p < alpha:
    print("Decision: Reject H0")
    print("Interpretation: Mean BMI differs significantly between smokers and non-smokers.")
else:
    print("Decision: Fail to reject H0")
    print("Interpretation: There is not enough evidence of a significant difference in mean BMI.")


## 9. Chi-square Test – Smoking Status and Diabetes

In [ ]:
contingency_table = pd.crosstab(
    df["smoking_status"],
    df["diabetes"]
)

chi2_stat, chi_p, chi_df, expected = stats.chi2_contingency(
    contingency_table
)

chi_critical = stats.chi2.ppf(
    1 - alpha,
    chi_df
)

print("Observed frequency table:")
display(contingency_table)

print(f"Chi-square statistic = {chi2_stat:.4f}")
print(f"Degrees of freedom = {chi_df}")
print(f"Critical chi-square value = {chi_critical:.4f}")
print(f"p-value = {chi_p:.8f}")

if chi_p < alpha:
    print("Decision: Reject H0")
    print("Interpretation: Smoking status and diabetes are significantly associated in this synthetic dataset.")
else:
    print("Decision: Fail to reject H0")
    print("Interpretation: There is not enough evidence of an association between smoking status and diabetes.")


## 10. ANOVA – Diabetes Rate Across Age Groups

In [ ]:
anova_groups = [
    group["diabetes"].astype(int).values
    for _, group in df.groupby("age_group", observed=True)
]

f_stat, anova_p = stats.f_oneway(*anova_groups)

k = len(anova_groups)
N = len(df)

f_critical = stats.f.ppf(
    1 - alpha,
    k - 1,
    N - k
)

diabetes_rates = (
    df.groupby("age_group", observed=True)["diabetes"]
      .mean()
      .mul(100)
)

print("Diabetes rate by age group:")
display(diabetes_rates.to_frame("Diabetes Rate (%)"))

print(f"F-statistic = {f_stat:.4f}")
print(f"Critical F-value = {f_critical:.4f}")
print(f"p-value = {anova_p:.8g}")

if anova_p < alpha:
    print("Decision: Reject H0")
    print("Interpretation: Diabetes rates differ significantly among the age groups.")
else:
    print("Decision: Fail to reject H0")
    print("Interpretation: There is not enough evidence that diabetes rates differ among age groups.")


## 11. Covariance – Age and BMI

In [ ]:
covariance = df[["age", "bmi"]].cov().loc["age", "bmi"]

print(f"Covariance between Age and BMI = {covariance:.4f}")

if covariance > 0:
    print("Interpretation: Age and BMI show positive co-movement.")
elif covariance < 0:
    print("Interpretation: Age and BMI show negative co-movement.")
else:
    print("Interpretation: The covariance is approximately zero.")


## 12. Correlation – Age and BMI

In [ ]:
correlation = df["age"].corr(df["bmi"])

print(f"Pearson correlation coefficient (r) = {correlation:.4f}")

if correlation > 0:
    strength = "positive"
elif correlation < 0:
    strength = "negative"
else:
    strength = "no"

print(f"Interpretation: The relationship is {strength} and its magnitude should be interpreted in context.")


## 13. Graphs

In [ ]:
# Diabetes rate by age group
diabetes_rates.plot(
    kind="bar",
    title="Diabetes Rate by Age Group",
    xlabel="Age Group",
    ylabel="Diabetes Rate (%)"
)
plt.tight_layout()
plt.show()


In [ ]:
# Smoking status vs diabetes
contingency_table.plot(
    kind="bar",
    title="Smoking Status vs Diabetes",
    xlabel="Smoking Status",
    ylabel="Number of Records"
)
plt.tight_layout()
plt.show()


In [ ]:
# Age vs BMI
plt.scatter(df["age"], df["bmi"])
plt.title("Age vs BMI")
plt.xlabel("Age")
plt.ylabel("BMI")
plt.tight_layout()
plt.show()


### Short notes for the report

For each test, I used a 5% significance level. If the p-value is below 0.05, I rejected H₀. Otherwise, I failed to reject H₀. The critical value was also checked wherever it was applicable.


## 14. Final Result

After running all the tests, I will compare each p-value with 0.05 and make the final decision about H₀. The main results are written below so they can be used in the assignment report.

**Note:** The data is generated for this practical, so the results are only for this sample and are not medical conclusions about real patients.
